# 03 — Scene Detection

This notebook investigates what environmental information can be extracted
from RGB video using a pretrained object detector.

## Pipeline

Video
->
YOLO
->
Object detections
->
Visualization

## Important limitation

Object detection alone does not provide:

- exact distance
- doorway width
- walkable path
- spatial difficulty
- recommended patient action

Those require additional spatial reasoning.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt

from cafog.perception import YOLODetector
from cafog.utils.video import iter_video_frames
from cafog.utils.visualization import draw_detections

DATA_DIR = Path("../data/raw")

videos = list(DATA_DIR.rglob("*.mp4"))

print(f"Videos found: {len(videos)}")

In [ ]:
detector = YOLODetector(
    model_path="yolo26n.pt",
    confidence=0.35,
)

print("YOLO detector initialized.")

In [ ]:
if videos:
    video_path = videos[0]

    samples = []

    for frame_index, frame in iter_video_frames(
        video_path,
        stride=30,
    ):
        detections = detector.detect(frame)

        annotated = draw_detections(
            frame,
            detections,
        )

        samples.append(
            (
                frame_index,
                annotated,
                detections,
            )
        )

        if len(samples) >= 6:
            break

In [ ]:
if samples:
    fig, axes = plt.subplots(
        2,
        3,
        figsize=(16, 9),
    )

    for ax, (
        frame_index,
        frame,
        detections,
    ) in zip(
        axes.flat,
        samples,
    ):
        frame_rgb = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB,
        )

        ax.imshow(frame_rgb)

        labels = [
            f"{d.class_name}: "
            f"{d.confidence:.2f}"
            for d in detections
        ]

        ax.set_title(
            f"Frame {frame_index}\n"
            + ", ".join(labels)
        )

        ax.axis("off")

    plt.tight_layout()
    plt.show()